In [ ]:
# =========================
# nb_ingest_castlebay
# =========================

# ---------- Parameters ----------
period = "2026-03"            # YYYY-MM
run_id = "manual_test_run"    # passed by nb_run_all in real run

dfm_id = "castlebay"
dfm_name = "Castlebay"

# ---------- Imports ----------
from pyspark.sql import functions as F
from pyspark.sql import Window
import pandas as pd
import re
import json

# ---------- Paths ----------
landing_path = f"Files/landing/period={period}/dfm={dfm_id}/source/"
fx_path = "Files/config/fx_rates.csv"
currency_map_path = "Files/config/currency_mapping.json"

print(f"[DEBUG] Landing path: {landing_path}")
print(f"[DEBUG] FX path: {fx_path}")
print(f"[DEBUG] Currency map path: {currency_map_path}")

# ---------- Helpers ----------
def q(s: str) -> str:
    return (s or "").replace("'", "''")

def norm(c: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", c.strip().lower()).strip("_")

def dec_col(c, p=28, s=10):
    return F.col(c).cast(f"decimal({p},{s})")

def write_audit(files_processed, rows_ingested, parse_errors_count, drift_events_count, status):
    print(f"[AUDIT] files_processed={files_processed}, rows_ingested={rows_ingested}, parse_errors={parse_errors_count}, status={status}")
    try:
        spark.sql(f"""
            INSERT INTO run_audit_log
            (run_id, period, dfm_id, files_processed, rows_ingested, parse_errors_count, drift_events_count, status, started_at, completed_at)
            VALUES
            ('{q(run_id)}','{q(period)}','{q(dfm_id)}',{int(files_processed)},{int(rows_ingested)},{int(parse_errors_count)},{int(drift_events_count)},'{q(status)}',current_timestamp(),current_timestamp())
        """)
    except Exception as e:
        print(f"[WARN] Failed to write audit log: {str(e)}")

def write_parse_errors(df_err):
    cols = set(df_err.columns)
    out = df_err

    if "source_file" not in cols:
        out = out.withColumn("source_file", F.lit("UNKNOWN"))
    if "row_number" not in cols:
        if "source_row_id" in cols:
            out = out.withColumn("row_number", F.regexp_extract(F.col("source_row_id").cast("string"), r"(\\d+)$", 1).cast("bigint"))
        else:
            out = out.withColumn("row_number", F.lit(None).cast("bigint"))
    if "column_name" not in cols:
        out = out.withColumn("column_name", F.lit(None).cast("string"))
    if "raw_value" not in cols:
        out = out.withColumn("raw_value", F.lit(None).cast("string"))
    if "error_type" not in cols:
        out = out.withColumn("error_type", F.lit("PARSE_ERROR"))
    if "error_message" not in cols:
        out = out.withColumn("error_message", F.lit("Unknown parse error"))

    try:
        err_count = out.count()
        print(f"[DEBUG] Writing {err_count} parse errors")
        (
            out.withColumn("period", F.lit(period))
               .withColumn("run_id", F.lit(run_id))
               .withColumn("dfm_id", F.lit(dfm_id))
               .withColumn("event_ts", F.current_timestamp())
               .select("period","run_id","dfm_id","source_file","row_number","column_name","raw_value","error_type","error_message","event_ts")
               .write.mode("append").saveAsTable("parse_errors")
        )
    except Exception as e:
        print(f"[WARN] Failed to write parse errors: {str(e)}")

# ---------- A3: File discovery ----------
print("[A3] Starting file discovery...")
try:
    entries = [f for f in mssparkutils.fs.ls(landing_path) if not f.isDir]
    print(f"[A3] Found {len(entries)} entries in landing path")
except Exception as ex:
    print(f"[A3] ERROR - Landing path not found or not accessible: {str(ex)}")
    write_audit(0,0,0,0,"NO_FILES")
    mssparkutils.notebook.exit("NO_FILES")

xlsx_files = [f for f in entries if f.name.lower().endswith(".xlsx")]
print(f"[A3] Found {len(xlsx_files)} XLSX files")
if len(xlsx_files) == 0:
    write_audit(0,0,0,0,"NO_FILES")
    mssparkutils.notebook.exit("NO_FILES")

# ---------- Load currency_mapping.json ----------
print("[CONFIG] Loading currency mapping...")
# Expect mapping from Currency Description -> ISO code
# Flexible parsing to handle either object/dict or list forms.
currency_mapping = {}

try:
    raw_json = mssparkutils.fs.head(currency_map_path, 5_000_000)
    cm = json.loads(raw_json)

    # Try common structures
    # 1) {"castlebay": {"US Dollar":"USD", ...}} or {"castlebay": [{"description":"US Dollar","iso":"USD"}]}
    if isinstance(cm, dict):
        if "castlebay" in cm:
            cb = cm["castlebay"]
            if isinstance(cb, dict):
                for k,v in cb.items():
                    currency_mapping[str(k).strip().lower()] = str(v).strip().upper()
            elif isinstance(cb, list):
                for item in cb:
                    if isinstance(item, dict):
                        d = item.get("description") or item.get("currency_description") or item.get("name")
                        i = item.get("iso") or item.get("iso_code") or item.get("code")
                        if d and i:
                            currency_mapping[str(d).strip().lower()] = str(i).strip().upper()
        else:
            # 2) flat dict {"US Dollar":"USD", ...}
            for k,v in cm.items():
                if isinstance(v, str):
                    currency_mapping[str(k).strip().lower()] = str(v).strip().upper()
    elif isinstance(cm, list):
        # 3) [{"description":"US Dollar","iso":"USD"}]
        for item in cm:
            if isinstance(item, dict):
                d = item.get("description") or item.get("currency_description") or item.get("name")
                i = item.get("iso") or item.get("iso_code") or item.get("code")
                if d and i:
                    currency_mapping[str(d).strip().lower()] = str(i).strip().upper()
    print(f"[CONFIG] Loaded {len(currency_mapping)} currency mappings")
except Exception as ex:
    print(f"[WARN] Currency mapping not loaded or malformed: {str(ex)}")
    # Continue; unknown descriptions will be flagged CURRENCY_UNKNOWN
    currency_mapping = {}

# broadcast mapping table
cm_rows = [(k, v) for k,v in currency_mapping.items()]
if len(cm_rows) > 0:
    cm_df = spark.createDataFrame(cm_rows, ["currency_description_norm", "local_currency_mapped"])
else:
    cm_df = spark.createDataFrame([], "currency_description_norm string, local_currency_mapped string")

# ---------- A4: Parse workbook(s) ----------
print("[A4] Starting workbook parsing...")
# SEDOL is optional (fallback when ISIN is missing)
required_cols = [
    "Account Name",
    "ISIN Code",
    "Quantity Held",
    "Price in Stock Currency",
    "Value in Market",
    "Currency Description"
]

parse_err_seed = []
pdf_all = []

for f in xlsx_files:
    print(f"[A4] Processing file: {f.name}")
    try:
        xls = pd.ExcelFile(f.path)
    except Exception as ex:
        print(f"[A4] ERROR opening {f.name}: {str(ex)}")
        parse_err_seed.append((f.name, "FILE", f"Unable to open workbook: {str(ex)}", ""))
        continue

    # Spec: include sheets with name containing "customer 1" or "customer 2"
    target_sheets = [s for s in xls.sheet_names if ("customer 1" in s.lower() or "customer 2" in s.lower())]
    print(f"[A4] {f.name}: Found {len(target_sheets)} target sheets: {target_sheets}")

    if len(target_sheets) == 0:
        parse_err_seed.append((f.name, "SHEET", "No target sheets (Customer 1/Customer 2) found", ""))
        continue

    # report_date from filename token DDMmmYY, e.g. 31Dec25
    m = re.search(r"(\d{2}[A-Za-z]{3}\d{2})", f.name)
    filename_date_token = m.group(1) if m else None
    print(f"[A4] {f.name}: Extracted date token: {filename_date_token}")

    for sh in target_sheets:
        try:
            # Spec: header row index 1-based = 3  => pandas header=2
            p = pd.read_excel(f.path, sheet_name=sh, header=2)
            cols = [str(c).strip() for c in p.columns]
            print(f"[A4] {f.name}:{sh}: Found {len(p)} rows, {len(cols)} columns")
            print(f"[A4] {f.name}:{sh}: Available columns: {cols}")
            
            if not all(c in cols for c in required_cols):
                missing = [c for c in required_cols if c not in cols]
                print(f"[A4] ERROR {f.name}:{sh}: Missing columns: {missing}")
                parse_err_seed.append((f.name, f"SHEET:{sh}", f"Missing required columns: {','.join(missing)}", ""))
                continue

            # Normalize SEDOL column name variations: "SEDOL Code" -> "SEDOL"
            if "SEDOL Code" in cols:
                print(f"[A4] {f.name}:{sh}: Renaming 'SEDOL Code' to 'SEDOL'")
                p.rename(columns={"SEDOL Code": "SEDOL"}, inplace=True)
            elif "SEDOL" not in cols:
                print(f"[A4] {f.name}:{sh}: SEDOL column missing, adding as NULL")
                p["SEDOL"] = None

            p["source_file"] = f.name
            p["source_sheet"] = sh
            p["filename_date_token"] = filename_date_token
            pdf_all.append(p)
            print(f"[A4] {f.name}:{sh}: Added {len(p)} rows to processing queue")
        except Exception as ex:
            print(f"[A4] ERROR parsing {f.name}:{sh}: {str(ex)}")
            parse_err_seed.append((f.name, f"SHEET:{sh}", f"Sheet parse failure: {str(ex)}", ""))
            continue

print(f"[A4] Completed: {len(pdf_all)} sheets parsed, {len(parse_err_seed)} parse errors")

if len(parse_err_seed) > 0:
    print(f"[A4] Writing {len(parse_err_seed)} parse errors to table...")
    write_parse_errors(spark.createDataFrame(parse_err_seed, ["source_file","source_row_id","error_message","raw_value"]))

if len(pdf_all) == 0:
    print("[FATAL] No valid data parsed from any sheet")
    write_audit(len(xlsx_files), 0, len(parse_err_seed), 0, "FAILED")
    mssparkutils.notebook.exit("FAILED")

pdf = pd.concat(pdf_all, ignore_index=True)
print(f"[A4] Concatenated dataframe: {len(pdf)} total rows")
pdf.columns = [norm(str(c)) for c in pdf.columns]

# Convert all columns to string to avoid type inference conflicts between sheets
print(f"[A4] Converting all columns to string for safe Spark ingestion")
for col in pdf.columns:
    pdf[col] = pdf[col].astype(str)

df_raw = spark.createDataFrame(pdf)

# source row id
w = Window.partitionBy("source_file", "source_sheet").orderBy(F.monotonically_increasing_id())
df_raw = df_raw.withColumn("_rn", F.row_number().over(w)) \
               .withColumn("source_row_id_raw", F.concat_ws(":", F.col("source_file"), F.col("source_sheet"), F.col("_rn").cast("string"))) \
               .drop("_rn")

# ---------- A5/A6: Map to canonical + type conversion ----------
print("[A5/A6] Mapping to canonical schema...")
# normalized source names expected:
# account_name, isin_code, sedol (optional), quantity_held, price_in_stock_currency, value_in_market, currency_description, filename_date_token

# Safely handle missing SEDOL column
if "sedol" not in df_raw.columns:
    df_raw = df_raw.withColumn("sedol", F.lit(None).cast("string"))

d = (
    df_raw
    .withColumn("policy_id", F.col("account_name").cast("string"))
    .withColumn("dfm_policy_id", F.col("account_name").cast("string"))
    .withColumn("isin", F.col("isin_code").cast("string"))
    .withColumn("sedol", F.col("sedol").cast("string"))
    .withColumn("security_id", F.coalesce(F.col("isin_code").cast("string"), F.col("sedol").cast("string")))
    .withColumn("holding_d", F.col("quantity_held").cast("double"))
    .withColumn("local_bid_price_d", F.col("price_in_stock_currency").cast("double"))
    .withColumn("bid_value_local_d", F.col("value_in_market").cast("double"))
    .withColumn("currency_description_raw", F.col("currency_description").cast("string"))
    .withColumn("currency_description_norm", F.lower(F.trim(F.col("currency_description").cast("string"))))
    .withColumn("report_date",
        F.to_date(F.col("filename_date_token"), "ddMMMyy")
    )
)

# join currency mapping
d = d.join(cm_df, on="currency_description_norm", how="left") \
     .withColumn("local_currency", F.col("local_currency_mapped")) \
     .drop("local_currency_mapped")

# fx rates
print("[A5/A6] Loading FX rates...")
try:
    fx = (
        spark.read.option("header", True).csv(fx_path)
        .withColumn("currency_u", F.upper(F.trim(F.col("currency"))))
        .withColumn("rate_to_gbp_d", F.col("rate_to_gbp").cast("double"))
        .select("currency_u", "rate_to_gbp_d")
    )
    fx_count = fx.count()
    print(f"[A5/A6] Loaded {fx_count} FX rates")
except Exception as ex:
    print(f"[WARN] FX rates not loaded: {str(ex)}")
    fx = spark.createDataFrame([], "currency_u string, rate_to_gbp_d double")

d = d.join(fx, F.col("local_currency") == F.col("currency_u"), "left")

# GBP rules
d = (
    d
    .withColumn(
        "fx_rate_out",
        F.when(F.col("local_currency") == "GBP", F.lit(1.0))
         .when(F.col("rate_to_gbp_d").isNotNull(), F.col("rate_to_gbp_d"))
         .otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "bid_value_gbp_d",
        F.when(F.col("local_currency") == "GBP", F.col("bid_value_local_d"))
         .when(F.col("rate_to_gbp_d").isNotNull(), F.col("bid_value_local_d") * F.col("rate_to_gbp_d"))
         .otherwise(F.lit(None).cast("double"))
    )
)

# defaults + flags
d = (
    d
    .withColumn("cash_value_gbp_d", F.lit(0.0))
    .withColumn("accrued_interest_gbp_d", F.lit(0.0))
    .withColumn("f_cash_defaulted", F.lit("CASH_DEFAULTED"))
    .withColumn("f_accrued_defaulted", F.lit("ACCRUED_DEFAULTED"))
    .withColumn("f_date_from_filename", F.when(F.col("report_date").isNotNull(), F.lit("DATE_FROM_FILENAME")))
    .withColumn("f_currency_unknown", F.when(F.col("local_currency").isNull(), F.lit("CURRENCY_UNKNOWN")))
    .withColumn("f_fx_not_available", F.when(F.col("bid_value_gbp_d").isNull(), F.lit("FX_NOT_AVAILABLE")))
    .withColumn(
        "data_quality_flags",
        F.expr("filter(array(f_cash_defaulted, f_accrued_defaulted, f_date_from_filename, f_currency_unknown, f_fx_not_available), x -> x is not null)")
    )
)

# parse errors for required fields
print("[A5/A6] Validating required fields...")
bad = (
    d
    .filter(F.col("policy_id").isNull() | F.col("report_date").isNull())
    .select(
        "source_file",
        F.col("source_row_id_raw").alias("source_row_id"),
        F.lit("Required field missing: policy_id/report_date").alias("error_message"),
        F.concat_ws("|", F.col("policy_id"), F.col("report_date").cast("string")).alias("raw_value")
    )
)

bad_count = bad.count()
print(f"[A5/A6] Found {bad_count} invalid rows (missing policy_id or report_date)")
if bad_count > 0:
    write_parse_errors(bad)

good = d.filter(F.col("policy_id").isNotNull() & F.col("report_date").isNotNull())
good_count = good.count()
print(f"[A5/A6] {good_count} rows passed validation")

# ---------- A7: source_row_id hash ----------
print("[A7] Generating source_row_id hash...")
good = good.withColumn(
    "source_row_id",
    F.sha2(
        F.concat_ws("|",
            F.lit(period),
            F.lit(dfm_id),
            F.col("policy_id"),
            F.coalesce(F.col("isin"), F.lit("")),
            F.col("report_date").cast("string"),
            F.col("holding_d").cast("string"),
            F.col("bid_value_local_d").cast("string"),
            F.col("source_file"),
            F.col("source_sheet"),
            F.col("source_row_id_raw")
        ),
        256
    )
)

# ---------- Canonical projection (exact contract) ----------
print("[A8] Projecting to canonical schema...")
canonical = (
    good
    .withColumn("period", F.lit(period))
    .withColumn("run_id", F.lit(run_id))
    .withColumn("dfm_id", F.lit(dfm_id))
    .withColumn("dfm_name", F.lit(dfm_name))
    .withColumn("policy_id_type", F.lit("DFM"))
    .withColumn("other_security_id", F.col("sedol").cast("string"))   # preserve SEDOL traceability
    .withColumn("id_type", F.when(F.col("isin").isNotNull(), F.lit("ISIN")).otherwise(F.lit("SEDOL")))
    .withColumn("asset_name", F.lit(None).cast("string"))
    .withColumn("ingested_at", F.current_timestamp())
    .select(
        "period",
        "run_id",
        "dfm_id",
        "dfm_name",
        "source_file",
        "source_sheet",
        "source_row_id",
        "policy_id",
        "policy_id_type",
        "dfm_policy_id",
        "security_id",
        "isin",
        "other_security_id",
        "id_type",
        "asset_name",
        dec_col("holding_d").alias("holding"),
        dec_col("local_bid_price_d").alias("local_bid_price"),
        "local_currency",
        dec_col("fx_rate_out").alias("fx_rate"),
        dec_col("cash_value_gbp_d").alias("cash_value_gbp"),
        dec_col("bid_value_gbp_d").alias("bid_value_gbp"),
        dec_col("accrued_interest_gbp_d").alias("accrued_interest_gbp"),
        "report_date",
        "ingested_at",
        "data_quality_flags"
    )
)

# de-duplicate in-run
print("[A8] Removing duplicates...")
canonical = canonical.dropDuplicates(["run_id","source_row_id"])
dedup_count = canonical.count()
print(f"[A8] {dedup_count} rows after deduplication")

# ---------- A8: Write ----------
print("[A8] Writing to canonical_holdings table...")
try:
    canonical.write.mode("append").saveAsTable("canonical_holdings")
    print("[A8] Successfully wrote to table")
except Exception as ex:
    print(f"[ERROR] Failed to write to canonical_holdings: {str(ex)}")
    write_audit(len(xlsx_files), 0, len(parse_err_seed) + bad_count, 0, "FAILED")
    mssparkutils.notebook.exit("FAILED")

rows_ingested = canonical.count()
print(f"[A8] Final rows ingested: {rows_ingested}")

# ---------- A9: Audit ----------
print("[A9] Writing audit log...")
status = "PARTIAL" if (bad_count + len(parse_err_seed)) > 0 else "OK"
write_audit(
    files_processed=len(xlsx_files),
    rows_ingested=rows_ingested,
    parse_errors_count=bad_count + len(parse_err_seed),
    drift_events_count=0,
    status=status
)

print(f"[SUCCESS] Notebook completed with status: {status}")

# ---------- A10: Exit ----------
mssparkutils.notebook.exit(status)
